# Block 0 — SpecMQuant reproduction gate

Runs the Block-0 correctness gate (IMPLEMENTATION_PLAN.md Phase 1): reproduce the
**direction** of SpecMQuant's finding — EAGLE speculative decoding's relative speedup
erodes when the target model is W4A16-quantized — on Llama-3-8B-Instruct, single-stream,
greedy, in vLLM.

**What it runs:** 8 cells = {FP16, W4A16-GPTQ-g128} x {no-spec, EAGLE} x {GSM8K, HumanEval},
4 server launches (one per model x spec combination), all driven by the repo harness.

**Requirements**
- **A100 runtime.** Runtime > Change runtime type > A100 GPU. Verify with the nvidia-smi
  cell below — don't trust the dropdown (PREREQ_RESULTS.md Check 1).
- **HF token** with access to `meta-llama/Meta-Llama-3-8B-Instruct` (gated: accept the
  license on the model page first). Store it as a Colab secret named `HF_TOKEN`
  (key icon in the left sidebar).

**Install recipe:** vLLM goes into an isolated `virtualenv` and runs as a subprocess —
NOT a bare pip install into the notebook kernel. Colab's preinstalled RAPIDS/torch stack
breaks a kernel-level vLLM install in confusing ways (PREREQ_RESULTS.md Check 6 recipe).

**Budget:** ~2–2.5 A100-hours end to end (two model downloads, 4 server startups at
~3 min each — Check 6 measured 172 s — plus ~40–70 min of timed measurement). **Note your
compute-units meter before and after: this run doubles as the Check-1 burn-rate calibration.**

In [ ]:
# 1. Verify the GPU actually assigned (don't trust the runtime-type dropdown)
!nvidia-smi
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
units_before = input("Compute-units balance right now (from the Colab usage panel): ")

In [ ]:
# 2. Get the repo
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo

In [ ]:
# 3. Isolated vLLM environment (Check 6 recipe: virtualenv, not stdlib venv;
#    ninja inside it for FlashInfer's JIT; ~6-8 min).
#    Pin: vllm==0.24.0 — confirmed to run this project's full stack (Check 6).
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)" 

In [ ]:
# 4. HF auth (gated Llama-3 weights)
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 5. Harness self-check inside the venv (no GPU needed, ~1 min).
#    If anything fails here, fix before burning GPU time.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

In [ ]:
# 6. Sanity: print the exact server commands the sweep will use (nothing launches)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/repro/repro_*.yaml" --results-dir results --dry-run

In [ ]:
# 7. THE SWEEP. Resumable: if Colab disconnects, re-run this cell — completed
# cells are skipped and only the interrupted one re-runs (a disconnect costs one cell).
# The harness launches `vllm serve` as a subprocess; PATH points into the venv so it
# finds the right vllm binary and ninja (FlashInfer JIT). Server logs: results/server_logs/.
# Rough timing: FP16 groups ~35-50 min, W4A16 groups ~20-35 min.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/repro/repro_*.yaml" --results-dir results

In [ ]:
# 8. Gate verdict
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m analysis.repro_gate results --targets configs/repro/reference_targets.yaml

In [ ]:
# 9. Preserve everything (results/ is git-ignored by design — archive it)
units_after = input("Compute-units balance now: ")
print("Units burned by this session:", units_before, "->", units_after)
!zip -qr block0_results.zip results
from google.colab import files
files.download("block0_results.zip")

## Reading the result

- **GATE: PASS** — direction reproduced. Archive `results/`, record the burn rate in
  PREREQ_RESULTS.md Check 1, proceed to Phase 2.
- **GATE: FAIL** — stop (PROJECT_SPEC.md §7.1). Debug order: (1) read
  `results/server_logs/*.log` for the failing group — EAGLE misconfig fails loudly at
  launch; (2) check tau in the report — tau ≈ 1 means the draft head isn't accepting
  anything (wrong draft/target pairing); (3) check the accuracy columns — greedy spec
  decode is output-preserving, so accuracy drift between spec-on/off cells means a
  harness bug, not a model property.
- **Magnitude warnings are expected, failures are not.** SpecMQuant's engine is bespoke
  C/CUDA with tree drafting (size 60); vLLM EAGLE is chain drafting
  (`num_speculative_tokens: 5`), and our W4A16 checkpoint is their non-rotation variant.
  Absolute speedups will differ; document the gap and its cause (EXPERIMENT_MATRIX.md §7,
  revised tolerance).

Known-good references (Figure 1a of arXiv 2505.22179, their engine, A100):
EAGLE-2 rel speedup 2.3x on FP16 -> 1.3x on W4A16; W4A16 quant-only 2.1x.
Provenance + gate thresholds: `configs/repro/reference_targets.yaml`.